In [1]:
from google.colab import drive
drive.mount

import sys
from pathlib import Path

!git -C /content/BraTS_thesis pull origin minvalue_norm

repo_src = Path("/content/BraTS_thesis/src")
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))


fatal: cannot change to '/content/BraTS_thesis': No such file or directory


In [ ]:
ZIP_PATH = Path("/content/drive/MyDrive/engineering_thesis/brats/processed_slices.zip")
print("Unzipping slices to local SSD")
!unzip -q {ZIP_PATH} -d /content/
print("Done")

In [ ]:
from brain_mri.data.dataset import create_dataloader

train_loader = create_dataloader(
    split_dir="content/processed/train",
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = create_dataloader(
    split_dir="content/processed/validate",
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Validate batches: {len(val_loader)}")

In [ ]:
import torch
from brain_mri.models.unet import UNet
from brain_mri.models.losses import DiceCELoss
from brain_mri.training import Trainer

model = UNet(in_channels=4, num_classes=4, base_features=64)

criterion = DiceCELoss()
optimizer = torch.optimizer.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
checkpoint_dir = Path("/content/drive/MyDrive/engineering_thesis/brats/checkpoints")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    checkpoint_dir=checkpoint_dir
)


In [ ]:
history = trainer.fit(epochs=30)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
# Loss function chart
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss function during training")
plt.legend()
plt.grid(True)
# Dice chart
plt.subplot(1, 2, 2)
plt.plot(history["val_mean_dice"], label="Val Mean Dice", color="green")
plt.xlabel("Epoch")
plt.ylabel("Dice coefficient")
plt.title("Mean dice on validate dataset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(checkpoint_dir / "learning_curves.png")
plt.show()